#Setting

In [ ]:
!unzip "/content/drive/MyDrive/Multi Encoder Vehicle Detection/splitted_dataset.zip" -d /content

Streaming output truncated to the last 5000 lines.
  inflating: /content/splitted_dataset/val/labels/e7c882b799fdea28c8a4932f08c75c3b.txt  
  inflating: /content/splitted_dataset/val/labels/a139297bb1f33301d46755e7c45e03e6.txt  
  inflating: /content/splitted_dataset/val/labels/72c48a6194747613299f407153d89e9a.txt  
  inflating: /content/splitted_dataset/val/labels/1d3c2518b9055468563ee24772807b30.txt  
  inflating: /content/splitted_dataset/val/labels/2b5de99ba91573d8e33b7e0f24549ca2.txt  
  inflating: /content/splitted_dataset/val/labels/2faea49e832eca84a59ae83975dcc49b.txt  
  inflating: /content/splitted_dataset/val/labels/89f4bedd8a2320cc011c7111d9a43f70.txt  
  inflating: /content/splitted_dataset/val/labels/4ec4c8c03a0355bb7cad5c6996a236c5.txt  
  inflating: /content/splitted_dataset/val/labels/0212d0924c457f71941b9c01eaaf511c.txt  
  inflating: /content/splitted_dataset/val/labels/3190dde032f7f07a9fd39450b2e096c6.txt  
  inflating: /content/splitted_dataset/val/labels/1f4c6e32d

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.7 MB/s eta 0:00:00


In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 18.6 MB/s eta 0:00:00


# Libraries

In [ ]:
!cp "/content/drive/MyDrive/Multi Encoder Vehicle Detection/model.py" "/content"

In [ ]:
# from ultralytics import RTDETR
import os
import warnings
# from model import CombinedModel
warnings.filterwarnings("ignore", message=".*grid_sampler_2d_backward_cuda.*")


# Yolo v12s

## optuna

In [ ]:
import json
from ultralytics import YOLO
def objective(trial):
    # Suggest hyperparameters
    lr0 = trial.suggest_float("lr0", 1e-5, 5e-4, log=True)
    lrf = trial.suggest_float("lrf", 0.001, 0.5)
    momentum = trial.suggest_float("momentum", 0.5, 0.98)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    imgsz = trial.suggest_categorical("imgsz", [640])
    batch = trial.suggest_categorical("batch", [4, 16])
    warmup_epochs = trial.suggest_int("warmup_epochs", 1, 5)
    optimizer_name = trial.suggest_categorical("optimizer", ["SGD", "Adam", "AdamW", "RAdam", "RMSprop"])
    # drop_path = trial.suggest_float("drop_path", 0.0, 0.2)

    model = YOLO('/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/YOLOv12/YOLOv12_3/weights/best.pt')

    save_dir = "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/YOLOv12/optuna"
    # Train the model
    results = model.train(
        data=os.path.join('splitted_dataset', 'optuna_data.yaml'),
        epochs=10,
        imgsz=imgsz,
        batch=batch,
        lr0=lr0,
        lrf=lrf,
        momentum=momentum,
        weight_decay=weight_decay,
        warmup_epochs=warmup_epochs,
        project=save_dir,
        name=f"trial_{trial.number}",
        optimizer=optimizer_name,
        exist_ok=True,
        verbose=False
    )

    # Extract performance metric
    # mAP50-95 is often used as the objective metric
    print(results.results_dict)
        # Save the dict to JSON
    json_path = os.path.join(save_dir, f"trial_{trial.number}_metrics.json")
    with open(json_path, "w") as f:
        json.dump(results.results_dict, f, indent=4)

    map50_95 = results.results_dict.get('metrics/mAP50-95(B)', 0.0)

    return map50_95


In [ ]:
import optuna

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best params:", study.best_params)
print("Best value:", study.best_value)


[I 2025-11-19 11:59:48,914] A new study created in memory with name: no-name-710d3456-d7b9-4fd9-8b7d-bd82939bb46b


Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=2.1928123993820695e-05, lrf=0.022581731950161117, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/YOLOv12/YOLOv12_3/weights/best.pt, momentum=0.7858022337860617, mosaic=1.0, multi_scale=False, name=tr

[I 2025-11-19 12:41:07,938] Trial 0 finished with value: 0.6282562805686495 and parameters: {'lr0': 2.1928123993820695e-05, 'lrf': 0.022581731950161117, 'momentum': 0.7858022337860617, 'weight_decay': 0.00046553774773717196, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 1, 'optimizer': 'SGD'}. Best is trial 0 with value: 0.6282562805686495.


{'metrics/precision(B)': 0.905150130607836, 'metrics/recall(B)': 0.8436758871193544, 'metrics/mAP50(B)': 0.9055248845168984, 'metrics/mAP50-95(B)': 0.6282562805686495, 'fitness': 0.6282562805686495}
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00012427716287903876, lrf=0.15265041481902292, mask_ratio=4, max_det=3

[I 2025-11-19 13:23:22,406] Trial 1 finished with value: 0.6255057760059473 and parameters: {'lr0': 0.00012427716287903876, 'lrf': 0.15265041481902292, 'momentum': 0.9299057114825284, 'weight_decay': 0.00018415010995695782, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 3, 'optimizer': 'AdamW'}. Best is trial 0 with value: 0.6282562805686495.


...

{'metrics/precision(B)': 0.9033158965078418, 'metrics/recall(B)': 0.8429506281161223, 'metrics/mAP50(B)': 0.9062924455754531, 'metrics/mAP50-95(B)': 0.6319205715632416, 'fitness': 0.6319205715632416}
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=5.830707861964347e-05, lrf=0.3733228146432894, mask_ratio=4, max_det=30

[I 2025-11-19 17:45:17,720] Trial 8 finished with value: 0.6276314385246062 and parameters: {'lr0': 5.830707861964347e-05, 'lrf': 0.3733228146432894, 'momentum': 0.6034820615223735, 'weight_decay': 7.031419095209061e-05, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 2, 'optimizer': 'SGD'}. Best is trial 5 with value: 0.633345351175094.


{'metrics/precision(B)': 0.906397399694322, 'metrics/recall(B)': 0.843746076205705, 'metrics/mAP50(B)': 0.9055788002424663, 'metrics/mAP50-95(B)': 0.6276314385246062, 'fitness': 0.6276314385246062}
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=1.2592414610426662e-05, lrf=0.05268552763278782, mask_ratio=4, max_det=30

[I 2025-11-19 18:26:38,250] Trial 9 finished with value: 0.6208964360532263 and parameters: {'lr0': 1.2592414610426662e-05, 'lrf': 0.05268552763278782, 'momentum': 0.7330726261265921, 'weight_decay': 5.927477728286016e-05, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 5, 'optimizer': 'Adam'}. Best is trial 5 with value: 0.633345351175094.


{'metrics/precision(B)': 0.896432101844667, 'metrics/recall(B)': 0.8396504075496702, 'metrics/mAP50(B)': 0.9009739788731006, 'metrics/mAP50-95(B)': 0.6208964360532263, 'fitness': 0.6208964360532263}
Best params: {'lr0': 9.990655492921155e-05, 'lrf': 0.4097449533118969, 'momentum': 0.6354662027280825, 'weight_decay': 0.00016565635745486164, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 2, 'optimizer': 'AdamW'}
Best value: 0.633345351175094


In [ ]:
try:
  optuna.visualization.plot_optimization_history(study).show()
  optuna.visualization.plot_param_importances(study).show()
except:
  pass

In [ ]:
print("Best params:", study.best_params)
print("Best value:", study.best_value)
best_params = study.best_params

Best params: {'lr0': 9.990655492921155e-05, 'lrf': 0.4097449533118969, 'momentum': 0.6354662027280825, 'weight_decay': 0.00016565635745486164, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 2, 'optimizer': 'AdamW'}
Best value: 0.633345351175094


## Fine tune on full

In [ ]:
model = YOLO('/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/YOLOv12/YOLOv12_3/weights/best.pt')

In [ ]:
best_params = {'lr0': 9.990655492921155e-05, 'lrf': 0.4097449533118969, 'momentum': 0.6354662027280825,
               'weight_decay': 0.00016565635745486164, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 2, 'optimizer': 'AdamW'}

In [ ]:

model.train(
    data=os.path.join('splitted_dataset', 'train_data.yaml'),
    epochs=30,
    imgsz=640,
    lr0=best_params["lr0"],
    lrf=best_params["lrf"],
    momentum=best_params["momentum"],
    weight_decay=best_params["weight_decay"],
    batch=best_params["batch"],
    optimizer = best_params["optimizer"],

    name='fine_tuned_yolov12',
    project='/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/YOLOv12/optuna_full',
)

Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/train_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=9.990655492921155e-05, lrf=0.4097449533118969, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/YOLOv12/YOLOv12_3/weights/best.pt, momentum=0.7392923951264895, mosaic=1.0, multi_scale=False, name=fine_

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7dd0983e22a0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

# Yolo World

## Optuna

In [ ]:
import json
from ultralytics import YOLO
from ultralytics import YOLOWorld
def objective(trial):
    # Suggest hyperparameters
    lr0 = trial.suggest_float("lr0", 1e-5, 5e-4, log=True)
    lrf = trial.suggest_float("lrf", 0.001, 0.5)
    momentum = trial.suggest_float("momentum", 0.5, 0.98)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    imgsz = trial.suggest_categorical("imgsz", [640])
    batch = trial.suggest_categorical("batch", [4, 16])
    warmup_epochs = trial.suggest_int("warmup_epochs", 1, 5)
    optimizer_name = trial.suggest_categorical("optimizer", ["SGD", "Adam", "AdamW", "RAdam", "RMSprop"])
    # drop_path = trial.suggest_float("drop_path", 0.0, 0.2)

    model = YOLOWorld('/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/YOLOWorld/YOLOWorld_2/weights/best.pt')

    save_dir = "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/YOLOWorld/optuna"
    # Train the model
    results = model.train(
        data=os.path.join('splitted_dataset', 'optuna_data.yaml'),
        epochs=10,
        imgsz=imgsz,
        batch=batch,
        lr0=lr0,
        lrf=lrf,
        momentum=momentum,
        weight_decay=weight_decay,
        warmup_epochs=warmup_epochs,
        project=save_dir,
        name=f"trial_{trial.number}",
        optimizer=optimizer_name,
        exist_ok=True,
        verbose=False
    )

    # Extract performance metric
    # mAP50-95 is often used as the objective metric
    print(results.results_dict)
        # Save the dict to JSON
    json_path = os.path.join(save_dir, f"trial_{trial.number}_metrics.json")
    with open(json_path, "w") as f:
        json.dump(results.results_dict, f, indent=4)

    map50_95 = results.results_dict.get('metrics/mAP50-95(B)', 0.0)

    return map50_95


In [ ]:
import optuna

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best params:", study.best_params)
print("Best value:", study.best_value)


[I 2025-11-20 11:49:16,840] A new study created in memory with name: no-name-953209f7-dd5f-40ff-8e9e-75b92cad7d03


Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=2.2294739828633784e-05, lrf=0.2356777463862784, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/YOLOWorld/YOLOWorld_2/weights/best.pt, momentum=0.9074863970386172, mosaic=1.0, multi_scale=False, name

[I 2025-11-20 12:21:52,567] Trial 0 finished with value: 0.6753583986968041 and parameters: {'lr0': 2.2294739828633784e-05, 'lrf': 0.2356777463862784, 'momentum': 0.9074863970386172, 'weight_decay': 0.0007471933817873102, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 4, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.6753583986968041.


{'metrics/precision(B)': 0.9150547170837893, 'metrics/recall(B)': 0.8586314118767554, 'metrics/mAP50(B)': 0.9192339799364386, 'metrics/mAP50-95(B)': 0.6753583986968041, 'fitness': 0.6753583986968041}
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=1.60838926040729e-05, lrf=0.1521266807885644, mask_ratio=4, max_det=300

[I 2025-11-20 12:59:33,040] Trial 1 finished with value: 0.6645003788411542 and parameters: {'lr0': 1.60838926040729e-05, 'lrf': 0.1521266807885644, 'momentum': 0.5999218671310982, 'weight_decay': 0.00017847462222479762, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 3, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.6753583986968041.


...

{'metrics/precision(B)': 0.0, 'metrics/recall(B)': 0.0, 'metrics/mAP50(B)': 0.0, 'metrics/mAP50-95(B)': 0.0, 'fitness': 0.0}
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=7.358028189434296e-05, lrf=0.1636984318359288, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Vehic

[I 2025-11-20 17:14:38,880] Trial 8 finished with value: 0.6741205225205242 and parameters: {'lr0': 7.358028189434296e-05, 'lrf': 0.1636984318359288, 'momentum': 0.5141701059861739, 'weight_decay': 0.00010990751552467443, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 4, 'optimizer': 'SGD'}. Best is trial 3 with value: 0.6773042973696036.


{'metrics/precision(B)': 0.9202097003477611, 'metrics/recall(B)': 0.8581985932887888, 'metrics/mAP50(B)': 0.9179585340092538, 'metrics/mAP50-95(B)': 0.6741205225205242, 'fitness': 0.6741205225205242}
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=8.135877059323838e-05, lrf=0.02838527953791179, mask_ratio=4, max_det=

[I 2025-11-20 17:48:59,104] Trial 9 finished with value: 0.6777496828582658 and parameters: {'lr0': 8.135877059323838e-05, 'lrf': 0.02838527953791179, 'momentum': 0.8803773427482824, 'weight_decay': 0.0001093279162798381, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 3, 'optimizer': 'Adam'}. Best is trial 9 with value: 0.6777496828582658.


{'metrics/precision(B)': 0.9126830398520408, 'metrics/recall(B)': 0.860411668358747, 'metrics/mAP50(B)': 0.9187365900716712, 'metrics/mAP50-95(B)': 0.6777496828582658, 'fitness': 0.6777496828582658}
Best params: {'lr0': 8.135877059323838e-05, 'lrf': 0.02838527953791179, 'momentum': 0.8803773427482824, 'weight_decay': 0.0001093279162798381, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 3, 'optimizer': 'Adam'}
Best value: 0.6777496828582658


In [ ]:
try:
  optuna.visualization.plot_optimization_history(study).show()
  optuna.visualization.plot_param_importances(study).show()
except:
  pass

In [ ]:
print("Best params:", study.best_params)
print("Best value:", study.best_value)
best_params = study.best_params

Best params: {'lr0': 8.135877059323838e-05, 'lrf': 0.02838527953791179, 'momentum': 0.8803773427482824, 'weight_decay': 0.0001093279162798381, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 3, 'optimizer': 'Adam'}
Best value: 0.6777496828582658


##Fine tuning on full

In [ ]:
model = YOLOWorld('/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/YOLOWorld/YOLOWorld_2/weights/best.pt')

In [ ]:
best_params = {'lr0': 8.135877059323838e-05, 'lrf': 0.02838527953791179,
               'momentum': 0.8803773427482824, 'weight_decay': 0.0001093279162798381,
               'imgsz': 640, 'batch': 16, 'warmup_epochs': 3, 'optimizer': 'Adam'}

In [ ]:

model.train(
    data=os.path.join('splitted_dataset', 'train_data.yaml'),
    epochs=30,
    imgsz=640,
    lr0=best_params["lr0"],
    lrf=best_params["lrf"],
    momentum=best_params["momentum"],
    weight_decay=best_params["weight_decay"],
    batch=best_params["batch"],
    optimizer = best_params["optimizer"],

    name='fine_tuned_YOLOWorld',
    project='/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/YOLOWorld/optuna_full',
)

Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/train_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=8.135877059323838e-05, lrf=0.02838527953791179, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/YOLOWorld/YOLOWorld_2/weights/best.pt, momentum=0.8803773427482824, mosaic=1.0, multi_scale=False, name

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79d7a93f7ce0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

# RTDETR Large

##Optuna

In [ ]:
import json
from ultralytics import RTDETR
def objective(trial):
    # Suggest hyperparameters
    lr0 = trial.suggest_float("lr0", 1e-5, 5e-4, log=True)
    lrf = trial.suggest_float("lrf", 0.001, 0.5)
    momentum = trial.suggest_float("momentum", 0.5, 0.98)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    imgsz = trial.suggest_categorical("imgsz", [640])
    batch = trial.suggest_categorical("batch", [4, 16])
    warmup_epochs = trial.suggest_int("warmup_epochs", 1, 5)
    optimizer_name = trial.suggest_categorical("optimizer", ["SGD", "Adam", "AdamW", "RAdam", "RMSprop"])
    # drop_path = trial.suggest_float("drop_path", 0.0, 0.2)

    model = RTDETR('/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/RTDTER/RTDTER4/weights/best.pt')

    save_dir = "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/RTDTER/optuna"
    # Train the model
    results = model.train(
        data=os.path.join('splitted_dataset', 'optuna_data.yaml'),
        epochs=10,
        imgsz=imgsz,
        batch=batch,
        lr0=lr0,
        lrf=lrf,
        momentum=momentum,
        weight_decay=weight_decay,
        warmup_epochs=warmup_epochs,
        project=save_dir,
        name=f"trial_{trial.number}",
        optimizer=optimizer_name,
        exist_ok=True,
        verbose=False
    )

    # Extract performance metric
    # mAP50-95 is often used as the objective metric
    print(results.results_dict)
        # Save the dict to JSON
    json_path = os.path.join(save_dir, f"trial_{trial.number}_metrics.json")
    with open(json_path, "w") as f:
        json.dump(results.results_dict, f, indent=4)

    map50_95 = results.results_dict.get('metrics/mAP50-95(B)', 0.0)

    return map50_95


In [ ]:
import optuna
import os

db_path = "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/RTDTER/optuna.db"

study = optuna.create_study(
    direction="maximize",
    study_name="rtdetr_tuning",
    storage=f"sqlite:///{db_path}",
    load_if_exists=True
)

study.optimize(objective, n_trials=5)

print("Trials done:", len(study.trials))
print("Best params:", study.best_params)
print("Best value:", study.best_value)


[I 2025-11-21 13:48:11,006] A new study created in RDB with name: rtdetr_tuning


Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001510916342488114, lrf=0.2148690233452203, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/RTDTER/RTDTER4/weights/best.pt, momentum=0.9703917599097354, mosaic=1.0, multi_scale=False, name=trial_0,

[I 2025-11-21 14:58:09,314] Trial 0 finished with value: 0.4412566900827318 and parameters: {'lr0': 0.0001510916342488114, 'lrf': 0.2148690233452203, 'momentum': 0.9703917599097354, 'weight_decay': 2.4499186365565714e-05, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 4, 'optimizer': 'AdamW'}. Best is trial 0 with value: 0.4412566900827318.


{'metrics/precision(B)': 0.5533485392487301, 'metrics/recall(B)': 0.7769282006678431, 'metrics/mAP50(B)': 0.6834363207286838, 'metrics/mAP50-95(B)': 0.4412566900827318, 'fitness': 0.4412566900827318}
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=1.3591620357276874e-05, lrf=0.0626291925547137, mask_ratio=4, max_det=3

[I 2025-11-21 16:10:09,093] Trial 1 finished with value: 0.6049651622183807 and parameters: {'lr0': 1.3591620357276874e-05, 'lrf': 0.0626291925547137, 'momentum': 0.8009096821684153, 'weight_decay': 1.4100689759078308e-05, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 5, 'optimizer': 'RAdam'}. Best is trial 1 with value: 0.6049651622183807.


...

{'metrics/precision(B)': 0.20393889045522218, 'metrics/recall(B)': 0.8327165470898072, 'metrics/mAP50(B)': 0.2782731118327145, 'metrics/mAP50-95(B)': 0.15090960346714963, 'fitness': 0.15090960346714963}
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=4.0228132212944845e-05, lrf=0.33781855451302467, mask_ratio=4, max_d

[I 2025-11-21 18:11:50,384] Trial 3 finished with value: 0.0 and parameters: {'lr0': 4.0228132212944845e-05, 'lrf': 0.33781855451302467, 'momentum': 0.8165108872835515, 'weight_decay': 1.6590269933971916e-05, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 4, 'optimizer': 'RMSprop'}. Best is trial 1 with value: 0.6049651622183807.


{'metrics/precision(B)': 0.0, 'metrics/recall(B)': 0.0, 'metrics/mAP50(B)': 0.0, 'metrics/mAP50-95(B)': 0.0, 'fitness': 0.0}
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00028957589756511536, lrf=0.17446485040635426, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Veh

[I 2025-11-21 19:01:39,381] Trial 4 finished with value: 0.6796435444544557 and parameters: {'lr0': 0.00028957589756511536, 'lrf': 0.17446485040635426, 'momentum': 0.7058344317537232, 'weight_decay': 0.00018009461892785287, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 3, 'optimizer': 'AdamW'}. Best is trial 4 with value: 0.6796435444544557.


{'metrics/precision(B)': 0.9162769909785322, 'metrics/recall(B)': 0.938815013487904, 'metrics/mAP50(B)': 0.9489471726991495, 'metrics/mAP50-95(B)': 0.6796435444544557, 'fitness': 0.6796435444544557}
Trials done: 5
Best params: {'lr0': 0.00028957589756511536, 'lrf': 0.17446485040635426, 'momentum': 0.7058344317537232, 'weight_decay': 0.00018009461892785287, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 3, 'optimizer': 'AdamW'}
Best value: 0.6796435444544557


In [ ]:
import optuna

db_path = "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/RTDTER/optuna.db"

study = optuna.load_study(
    study_name="rtdetr_tuning",
    storage=f"sqlite:///{db_path}"
)

study.optimize(objective, n_trials=5)

print("Total trials:", len(study.trials))


New https://pypi.org/project/ultralytics/8.3.230 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0004954412722113413, lrf=0.2874216127241907, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/RTD

[I 2025-11-21 20:38:53,901] Trial 8 finished with value: 0.5781070715705054 and parameters: {'lr0': 0.0004954412722113413, 'lrf': 0.2874216127241907, 'momentum': 0.9267171956202167, 'weight_decay': 0.0002139663042627432, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 4, 'optimizer': 'AdamW'}. Best is trial 4 with value: 0.6796435444544557.


{'metrics/precision(B)': 0.8174509093153117, 'metrics/recall(B)': 0.9086539206446762, 'metrics/mAP50(B)': 0.8687773595692739, 'metrics/mAP50-95(B)': 0.5781070715705054, 'fitness': 0.5781070715705054}
New https://pypi.org/project/ultralytics/8.3.230 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False,

[I 2025-11-21 21:27:14,759] Trial 9 finished with value: 0.7144189527725956 and parameters: {'lr0': 0.00039462947108956354, 'lrf': 0.44188676177939823, 'momentum': 0.6170324018205549, 'weight_decay': 0.00020058894588688875, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 4, 'optimizer': 'SGD'}. Best is trial 9 with value: 0.7144189527725956.


New https://pypi.org/project/ultralytics/8.3.230 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00041649140953646866, lrf=0.4131613349574731, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/R

...

{'metrics/precision(B)': 0.9240744806675704, 'metrics/recall(B)': 0.9266820393970971, 'metrics/mAP50(B)': 0.9371105299199267, 'metrics/mAP50-95(B)': 0.6248996210896763, 'fitness': 0.6248996210896763}
New https://pypi.org/project/ultralytics/8.3.230 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False,

[I 2025-11-21 23:04:01,478] Trial 11 finished with value: 0.6326053962090089 and parameters: {'lr0': 6.455061887977484e-05, 'lrf': 0.3220304644556169, 'momentum': 0.7413840004494897, 'weight_decay': 3.896176809761741e-05, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 2, 'optimizer': 'RAdam'}. Best is trial 9 with value: 0.7144189527725956.


New https://pypi.org/project/ultralytics/8.3.230 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=1.0447641093431063e-05, lrf=0.36564246683413043, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/R

[I 2025-11-22 00:14:31,649] Trial 12 finished with value: 0.6499322865268489 and parameters: {'lr0': 1.0447641093431063e-05, 'lrf': 0.36564246683413043, 'momentum': 0.6134390876219622, 'weight_decay': 0.00019688026232362097, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 3, 'optimizer': 'SGD'}. Best is trial 9 with value: 0.7144189527725956.


{'metrics/precision(B)': 0.9489949761437928, 'metrics/recall(B)': 0.9207967246408725, 'metrics/mAP50(B)': 0.9487628888348403, 'metrics/mAP50-95(B)': 0.6499322865268489, 'fitness': 0.6499322865268489}
Total trials: 13


In [ ]:
try:
  optuna.visualization.plot_optimization_history(study).show()
  optuna.visualization.plot_param_importances(study).show()
except:
  pass

In [ ]:
print("Best params:", study.best_params)
print("Best value:", study.best_value)
best_params = study.best_params

Best params: {'lr0': 0.00039462947108956354, 'lrf': 0.44188676177939823, 'momentum': 0.6170324018205549, 'weight_decay': 0.00020058894588688875, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 4, 'optimizer': 'SGD'}
Best value: 0.7144189527725956


## Fine Tunning on full

In [ ]:
import json
from ultralytics import RTDETR
model = RTDETR('/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/RTDTER/RTDTER4/weights/best.pt')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
best_params = {'lr0': 0.00039462947108956354, 'lrf': 0.44188676177939823,
               'momentum': 0.6170324018205549, 'weight_decay': 0.00020058894588688875,
               'imgsz': 640, 'batch': 16, 'warmup_epochs': 4, 'optimizer': 'SGD'}

In [ ]:

model.train(
    data=os.path.join('splitted_dataset', 'train_data.yaml'),
    epochs=30,
    imgsz=640,
    lr0=best_params["lr0"],
    lrf=best_params["lrf"],
    momentum=best_params["momentum"],
    weight_decay=best_params["weight_decay"],
    batch=best_params["batch"],
    optimizer = best_params["optimizer"],

    name='fine_tuned_RTDTER',
    project='/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/RTDTER/optuna_full',
)

Ultralytics 8.3.230 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/train_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00039462947108956354, lrf=0.44188676177939823, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/RTDTER/RTDTER4/weights/best.pt, momentum=0.6170324018205549, mosaic=1.0, multi_scale=False, name=fine_

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x789bd06e0da0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

# MobileSam + Clip

##Optuna

In [ ]:
import json
from model import CombinedModel
from ultralytics import RTDETR
def objective(trial):
    # Suggest hyperparameters
    lr0 = trial.suggest_float("lr0", 1e-5, 5e-4, log=True)
    lrf = trial.suggest_float("lrf", 0.001, 0.5)
    momentum = trial.suggest_float("momentum", 0.5, 0.98)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    imgsz = trial.suggest_categorical("imgsz", [640])
    batch = trial.suggest_categorical("batch", [4, 16])
    warmup_epochs = trial.suggest_int("warmup_epochs", 1, 5)
    optimizer_name = trial.suggest_categorical("optimizer", ["SGD", "Adam", "AdamW", "RAdam", "RMSprop"])
    # drop_path = trial.suggest_float("drop_path", 0.0, 0.2)

    model = RTDETR('/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/Hybrid_MobileSam_Clip3/weights/best.pt')
    hybrid_encoder = CombinedModel(features=['sam', 'clip'], load_FusionGate=True, load_encoders=True)
    model.model.model[0]= hybrid_encoder

    save_dir = "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/optuna"
    # Train the model
    results = model.train(
        data=os.path.join('splitted_dataset', 'optuna_data.yaml'),
        epochs=10,
        imgsz=imgsz,
        batch=batch,
        lr0=lr0,
        lrf=lrf,
        momentum=momentum,
        weight_decay=weight_decay,
        warmup_epochs=warmup_epochs,
        project=save_dir,
        name=f"trial_{trial.number}",
        optimizer=optimizer_name,
        exist_ok=True,
        verbose=False
    )

    # Extract performance metric
    # mAP50-95 is often used as the objective metric
    print(results.results_dict)
        # Save the dict to JSON
    json_path = os.path.join(save_dir, f"trial_{trial.number}_metrics.json")
    with open(json_path, "w") as f:
        json.dump(results.results_dict, f, indent=4)

    map50_95 = results.results_dict.get('metrics/mAP50-95(B)', 0.0)

    return map50_95


In [ ]:
import optuna
import os

db_path = "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/optuna.db"

study = optuna.create_study(
    direction="maximize",
    study_name="hybrid_tuning",
    storage=f"sqlite:///{db_path}",
    load_if_exists=True
)

study.optimize(objective, n_trials=5)

print("Trials done:", len(study.trials))
print("Best params:", study.best_params)
print("Best value:", study.best_value)


[I 2025-11-24 10:12:26,506] A new study created in RDB with name: hybrid_tuning


Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

{'metrics/precision(B)': 0.9577988687387013, 'metrics/recall(B)': 0.9456709411204192, 'metrics/mAP50(B)': 0.9617115505761819, 'metrics/mAP50-95(B)': 0.6429992422356201, 'fitness': 0.6429992422356201}
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_widt

[I 2025-11-24 14:14:26,976] Trial 3 finished with value: 0.010393318395696727 and parameters: {'lr0': 0.000136484186826284, 'lrf': 0.2507999816698113, 'momentum': 0.8685694881399806, 'weight_decay': 0.00011598458222332858, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 2, 'optimizer': 'RMSprop'}. Best is trial 2 with value: 0.6429992422356201.


{'metrics/precision(B)': 0.006659950749944034, 'metrics/recall(B)': 0.21750255885363357, 'metrics/mAP50(B)': 0.05513356058712953, 'metrics/mAP50-95(B)': 0.010393318395696727, 'fitness': 0.010393318395696727}
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, l

[I 2025-11-24 15:25:54,874] Trial 4 finished with value: 0.5528204376976688 and parameters: {'lr0': 0.00011994037810462294, 'lrf': 0.43458343770798546, 'momentum': 0.9161747233507812, 'weight_decay': 1.3852828977648227e-05, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 1, 'optimizer': 'SGD'}. Best is trial 2 with value: 0.6429992422356201.


{'metrics/precision(B)': 0.8792858463804063, 'metrics/recall(B)': 0.860418607158921, 'metrics/mAP50(B)': 0.8887241087848841, 'metrics/mAP50-95(B)': 0.5528204376976688, 'fitness': 0.5528204376976688}
Trials done: 5
Best params: {'lr0': 0.0001834673993800264, 'lrf': 0.062248018142410474, 'momentum': 0.5815273899054546, 'weight_decay': 8.483503650054716e-05, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 3, 'optimizer': 'RAdam'}
Best value: 0.6429992422356201


In [ ]:
import optuna

db_path = "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/optuna.db"

study = optuna.load_study(
    study_name="hybrid_tuning",
    storage=f"sqlite:///{db_path}"
)

study.optimize(objective, n_trials=5)

print("Total trials:", len(study.trials))


Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00037881344090538694, lrf=0.228588601837636, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/Hybrid_MobileSam_Clip3/weights/best.pt, momentum=0.5833580138120841, mos

[I 2025-11-24 16:38:41,103] Trial 6 finished with value: 0.6468989657717407 and parameters: {'lr0': 0.00037881344090538694, 'lrf': 0.228588601837636, 'momentum': 0.5833580138120841, 'weight_decay': 0.00030934401335096527, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 5, 'optimizer': 'RAdam'}. Best is trial 6 with value: 0.6468989657717407.


Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=1.0232476606539399e-05, lrf=0.16157262387838497, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Cli

...

{'metrics/precision(B)': 0.917441910285933, 'metrics/recall(B)': 0.9230357738116877, 'metrics/mAP50(B)': 0.9428324258040635, 'metrics/mAP50-95(B)': 0.6038256237995088, 'fitness': 0.6038256237995088}
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width

[I 2025-11-24 20:11:41,487] Trial 9 finished with value: 0.6222427792841115 and parameters: {'lr0': 0.0004443214652295646, 'lrf': 0.21689950863300875, 'momentum': 0.7197828209964454, 'weight_decay': 0.00012808149093458535, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 1, 'optimizer': 'Adam'}. Best is trial 6 with value: 0.6468989657717407.


{'metrics/precision(B)': 0.9362735359671743, 'metrics/recall(B)': 0.9425952205633279, 'metrics/mAP50(B)': 0.9572382095290239, 'metrics/mAP50-95(B)': 0.6222427792841115, 'fitness': 0.6222427792841115}
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_widt

[I 2025-11-24 21:22:21,815] Trial 10 finished with value: 0.4678745193681732 and parameters: {'lr0': 6.636182800265251e-05, 'lrf': 0.38930441377076785, 'momentum': 0.860770752619156, 'weight_decay': 0.0005384434337419616, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 3, 'optimizer': 'SGD'}. Best is trial 6 with value: 0.6468989657717407.


{'metrics/precision(B)': 0.8276732485671825, 'metrics/recall(B)': 0.796811280582933, 'metrics/mAP50(B)': 0.8141929735244753, 'metrics/mAP50-95(B)': 0.4678745193681732, 'fitness': 0.4678745193681732}
Total trials: 11


In [ ]:
try:
  optuna.visualization.plot_optimization_history(study).show()
  optuna.visualization.plot_param_importances(study).show()
except:
  pass

In [ ]:
print("Best params:", study.best_params)
print("Best value:", study.best_value)
best_params = study.best_params

Best params: {'lr0': 0.00037881344090538694, 'lrf': 0.228588601837636, 'momentum': 0.5833580138120841, 'weight_decay': 0.00030934401335096527, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 5, 'optimizer': 'RAdam'}
Best value: 0.6468989657717407


## Fine tune on full

In [ ]:
import json
from model import CombinedModel
from ultralytics import RTDETR
model = RTDETR('/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/Hybrid_MobileSam_Clip3/weights/best.pt')
hybrid_encoder = CombinedModel(features=['sam', 'clip'], load_FusionGate=True, load_encoders=True)
model.model.model[0]= hybrid_encoder

Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
best_params = {'lr0': 0.00037881344090538694, 'lrf': 0.228588601837636,
               'momentum': 0.5833580138120841, 'weight_decay': 0.00030934401335096527, 'imgsz': 640,
               'batch': 16, 'warmup_epochs': 5, 'optimizer': 'RAdam'}

In [ ]:

model.train(
    data=os.path.join('splitted_dataset', 'train_data.yaml'),
    epochs=30,
    imgsz=640,
    lr0=best_params["lr0"],
    lrf=best_params["lrf"],
    momentum=best_params["momentum"],
    weight_decay=best_params["weight_decay"],
    batch=best_params["batch"],
    optimizer = best_params["optimizer"],

    name='fine_tuned_hybrid',
    project='/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/optuna_full',
)

Ultralytics 8.3.231 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/train_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00037881344090538694, lrf=0.228588601837636, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/Hybrid_MobileSam_Clip3/weights/best.pt, momentum=0.5833580138120841, mos

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x781b7c58b110>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
import torch
torch.save(model.model.state_dict(), "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/full_model.pth")
# torch.save({
#     "model": model.model.state_dict(),
#     "optimizer": model.trainer.optimizer.state_dict(),
#     "epoch": model.trainer.epoch
# }, "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/full_model.pth")


### more epochs

In [ ]:
import json
import torch
from model import CombinedModel
from ultralytics import RTDETR

In [ ]:
model = RTDETR('/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/optuna_full/fine_tuned_hybrid/weights/best.pt')
hybrid_encoder = CombinedModel(features=['sam', 'clip'], load_FusionGate=True, load_encoders=True)
model.model.model[0]= hybrid_encoder

Ultralytics 8.3.232 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)


In [ ]:
state_dict = torch.load("/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/full_model.pth")
model.load_state_dict(state_dict, strict=False)

In [ ]:
best_params = {'lr0': 0.00037881344090538694, 'lrf': 0.228588601837636,
               'momentum': 0.5833580138120841, 'weight_decay': 0.00030934401335096527, 'imgsz': 640,
               'batch': 16, 'warmup_epochs': 5, 'optimizer': 'RAdam'}

In [ ]:
model.train(
    data=os.path.join('splitted_dataset', 'train_data.yaml'),
    epochs=30,
    imgsz=640,
    lr0=best_params["lr0"],
    lrf=best_params["lrf"],
    momentum=best_params["momentum"],
    weight_decay=best_params["weight_decay"],
    batch=best_params["batch"],
    optimizer = best_params["optimizer"],

    name='fine_tuned_hybrid',
    project='/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/optuna_full',
)

Ultralytics 8.3.232 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/train_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00037881344090538694, lrf=0.228588601837636, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/optuna_full/fine_tuned_hybrid/weights/best.pt, momentum=0.58335801381208

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c59474b4950>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
import torch
# torch.save(model.state_dict(), "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/full_model.pth")
torch.save({
    "model": model.model.state_dict(),
    "optimizer": model.trainer.optimizer.state_dict(),
    "epoch": model.trainer.epoch
}, "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/full_model2.pth")


# ConvNext-Small

In [ ]:
import torch
import torch.nn as nn
import timm

class FusionConvNeXtSmall(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.f = -1
        self.i=0

        # 🔹 ConvNeXt-S backbone
        self.backbone = timm.create_model(
            "convnext_small.fb_in22k_ft_in1k",
            pretrained=pretrained,
            features_only=True,
            out_indices=(0, 1, 2, 3)   # all 4 stages
        )

        # Get output channels of each stage
        feat_channels = self.backbone.feature_info.channels()  # e.g. [96, 192, 384, 768]

        # 🔹 Per-stage projection → (B,128,160,160)
        self.stage_proj = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(c, 128, kernel_size=1),
                nn.Upsample(size=(160,160), mode="bilinear", align_corners=False)
            )
            for c in feat_channels
        ])

        # 🔹 After concat: fuse to 48 channels
        self.fuse_conv = nn.Conv2d(128 * len(feat_channels), 48, kernel_size=1)

    def forward(self, x):
        # 1. Extract features
        feats = self.backbone(x)  # list of 4 feature maps

        # 2. Project + resize
        proj_feats = [proj(f) for f, proj in zip(feats, self.stage_proj)]  # each → B,128,160,160

        # 3. Concatenate along channel dim
        fused = torch.cat(proj_feats, dim=1)  # B,128*4,160,160

        # 4. Final conv → B,48,160,160
        out = self.fuse_conv(fused)
        return out

## Optuna

In [ ]:
import json
from model import CombinedModel
from ultralytics import RTDETR

def objective(trial):
    # Suggest hyperparameters
    lr0 = trial.suggest_float("lr0", 1e-5, 5e-4, log=True)
    lrf = trial.suggest_float("lrf", 0.001, 0.5)
    momentum = trial.suggest_float("momentum", 0.5, 0.98)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    imgsz = trial.suggest_categorical("imgsz", [640])
    batch = trial.suggest_categorical("batch", [4, 16])
    warmup_epochs = trial.suggest_int("warmup_epochs", 1, 5)
    optimizer_name = trial.suggest_categorical("optimizer", ["SGD", "Adam", "AdamW", "RAdam", "RMSprop"])
    # drop_path = trial.suggest_float("drop_path", 0.0, 0.2)

    model = RTDETR('rtdetr-l.pt')
    hybrid_encoder = FusionConvNeXtSmall(pretrained=True)
    model.model.model[0]= hybrid_encoder

    save_dir = "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_ConvNext/optuna"
    # Train the model
    results = model.train(
        data=os.path.join('splitted_dataset', 'optuna_data.yaml'),
        epochs=10,
        imgsz=imgsz,
        batch=batch,
        lr0=lr0,
        lrf=lrf,
        momentum=momentum,
        weight_decay=weight_decay,
        warmup_epochs=warmup_epochs,
        project=save_dir,
        name=f"trial_{trial.number}",
        optimizer=optimizer_name,
        exist_ok=True,
        verbose=False
    )

    # Extract performance metric
    # mAP50-95 is often used as the objective metric
    print(results.results_dict)
        # Save the dict to JSON
    json_path = os.path.join(save_dir, f"trial_{trial.number}_metrics.json")
    with open(json_path, "w") as f:
        json.dump(results.results_dict, f, indent=4)

    map50_95 = results.results_dict.get('metrics/mAP50-95(B)', 0.0)

    return map50_95


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
import optuna
import os

db_path = "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_ConvNext/optuna.db"

study = optuna.create_study(
    direction="maximize",
    study_name="hybrid_tuning",
    storage=f"sqlite:///{db_path}",
    load_if_exists=True
)

study.optimize(objective, n_trials=5)

print("Trials done:", len(study.trials))
print("Best params:", study.best_params)
print("Best value:", study.best_value)


[I 2025-12-16 17:18:38,520] A new study created in RDB with name: hybrid_tuning


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/201M [00:00<?, ?B/s]

Ultralytics 8.3.239 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0004238757723592467, lrf=0.24346380330907622, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.7331644169868371, mosaic=1.0, multi_scale=False, name=trial_0, nbs=64, nms=False, opset=None, optimize=False, optimizer=RMSprop, overlap_mask=T

[I 2025-12-16 18:30:56,892] Trial 0 finished with value: 0.0 and parameters: {'lr0': 0.0004238757723592467, 'lrf': 0.24346380330907622, 'momentum': 0.7331644169868371, 'weight_decay': 0.0007742838593540302, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 2, 'optimizer': 'RMSprop'}. Best is trial 0 with value: 0.0.


...

{'metrics/precision(B)': 0.4097307203720658, 'metrics/recall(B)': 0.5487768218195432, 'metrics/mAP50(B)': 0.4116490081440116, 'metrics/mAP50-95(B)': 0.20003751695622682, 'fitness': 0.20003751695622682}
Ultralytics 8.3.239 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=4.2522155278423587e-05, lrf=0.036318644072215314, mask_ratio=4, max_d

[I 2025-12-16 21:45:48,401] Trial 3 finished with value: 0.0177970965794537 and parameters: {'lr0': 4.2522155278423587e-05, 'lrf': 0.036318644072215314, 'momentum': 0.8751295256005547, 'weight_decay': 0.00027769081634038324, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 3, 'optimizer': 'SGD'}. Best is trial 2 with value: 0.20003751695622682.


{'metrics/precision(B)': 0.5718906045485926, 'metrics/recall(B)': 0.12238631378856558, 'metrics/mAP50(B)': 0.048326032467947665, 'metrics/mAP50-95(B)': 0.0177970965794537, 'fitness': 0.0177970965794537}
Ultralytics 8.3.239 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=2.419240888187987e-05, lrf=0.15366585718972461, mask_ratio=4, max_d

[I 2025-12-16 22:35:26,399] Trial 4 finished with value: 0.3275714679812023 and parameters: {'lr0': 2.419240888187987e-05, 'lrf': 0.15366585718972461, 'momentum': 0.6859223668807861, 'weight_decay': 0.00011350903989016466, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 4, 'optimizer': 'AdamW'}. Best is trial 4 with value: 0.3275714679812023.


{'metrics/precision(B)': 0.6228383235844794, 'metrics/recall(B)': 0.7427413310511186, 'metrics/mAP50(B)': 0.6456845257156666, 'metrics/mAP50-95(B)': 0.3275714679812023, 'fitness': 0.3275714679812023}
Trials done: 5
Best params: {'lr0': 2.419240888187987e-05, 'lrf': 0.15366585718972461, 'momentum': 0.6859223668807861, 'weight_decay': 0.00011350903989016466, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 4, 'optimizer': 'AdamW'}
Best value: 0.3275714679812023


In [ ]:
import optuna

db_path = "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_ConvNext/optuna.db"

study = optuna.load_study(
    study_name="hybrid_tuning",
    storage=f"sqlite:///{db_path}"
)

study.optimize(objective, n_trials=5)

print("Total trials:", len(study.trials))


Ultralytics 8.3.239 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=6.882855138694592e-05, lrf=0.1672990838563836, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.7561724547771908, mosaic=1.0, multi_scale=False, name=trial_5, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, 

[I 2025-12-16 23:48:16,432] Trial 5 finished with value: 0.0061002058189445745 and parameters: {'lr0': 6.882855138694592e-05, 'lrf': 0.1672990838563836, 'momentum': 0.7561724547771908, 'weight_decay': 0.0003271142312059666, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 3, 'optimizer': 'SGD'}. Best is trial 4 with value: 0.3275714679812023.


{'metrics/precision(B)': 0.5402287953778653, 'metrics/recall(B)': 0.07987278841935956, 'metrics/mAP50(B)': 0.01759406774760973, 'metrics/mAP50-95(B)': 0.0061002058189445745, 'fitness': 0.0061002058189445745}
Ultralytics 8.3.239 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=4.386923040313738e-05, lrf=0.09514580166659296, mask_ratio=4, m

[I 2025-12-17 01:01:39,115] Trial 6 finished with value: 0.328775185047684 and parameters: {'lr0': 4.386923040313738e-05, 'lrf': 0.09514580166659296, 'momentum': 0.958082232193125, 'weight_decay': 2.8735395765102986e-05, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 4, 'optimizer': 'RAdam'}. Best is trial 6 with value: 0.328775185047684.


{'metrics/precision(B)': 0.6790158406232956, 'metrics/recall(B)': 0.6921452485671562, 'metrics/mAP50(B)': 0.6811546279949313, 'metrics/mAP50-95(B)': 0.328775185047684, 'fitness': 0.328775185047684}
Ultralytics 8.3.239 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=2.8135802639623318e-05, lrf=0.3575501730158067, mask_ratio=4, max_det=30

...

{'metrics/precision(B)': 0.6541703776133516, 'metrics/recall(B)': 0.7499084959528668, 'metrics/mAP50(B)': 0.6830441810804646, 'metrics/mAP50-95(B)': 0.3470849751987492, 'fitness': 0.3470849751987492}
Ultralytics 8.3.239 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00019916791139334698, lrf=0.1234209232643537, mask_ratio=4, max_det=

[I 2025-12-17 02:41:34,274] Trial 8 finished with value: 0.0 and parameters: {'lr0': 0.00019916791139334698, 'lrf': 0.1234209232643537, 'momentum': 0.7157368431970323, 'weight_decay': 0.0007853768086619303, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 4, 'optimizer': 'RMSprop'}. Best is trial 7 with value: 0.3470849751987492.


Ultralytics 8.3.239 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/optuna_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00018495357414927133, lrf=0.3016320063611268, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.6082449295870077, mosaic=1.0, multi_scale=False, name=trial_9, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True,

[I 2025-12-17 03:55:07,590] Trial 9 finished with value: 0.025203676516486452 and parameters: {'lr0': 0.00018495357414927133, 'lrf': 0.3016320063611268, 'momentum': 0.6082449295870077, 'weight_decay': 8.393769161655501e-05, 'imgsz': 640, 'batch': 4, 'warmup_epochs': 5, 'optimizer': 'SGD'}. Best is trial 7 with value: 0.3470849751987492.


{'metrics/precision(B)': 0.5924034484873781, 'metrics/recall(B)': 0.16427840327533266, 'metrics/mAP50(B)': 0.06656494328578537, 'metrics/mAP50-95(B)': 0.025203676516486452, 'fitness': 0.025203676516486452}
Total trials: 10


In [ ]:
try:
  optuna.visualization.plot_optimization_history(study).show()
  optuna.visualization.plot_param_importances(study).show()
except:
  pass

In [ ]:
print("Best params:", study.best_params)
print("Best value:", study.best_value)
best_params = study.best_params

Best params: {'lr0': 2.8135802639623318e-05, 'lrf': 0.3575501730158067, 'momentum': 0.8114575443568205, 'weight_decay': 1.3657965520977741e-05, 'imgsz': 640, 'batch': 16, 'warmup_epochs': 2, 'optimizer': 'AdamW'}
Best value: 0.3470849751987492


In [ ]:
import json
from model import CombinedModel
from ultralytics import RTDETR
model = RTDETR('rtdetr-l.pt')
hybrid_encoder = FusionConvNeXtSmall(pretrained=True)
model.model.model[0]= hybrid_encoder

In [ ]:
best_params = {'lr0': 2.8135802639623318e-05, 'lrf': 0.3575501730158067,
               'momentum': 0.8114575443568205, 'weight_decay': 1.3657965520977741e-05,
               'imgsz': 640, 'batch': 16, 'warmup_epochs': 2, 'optimizer': 'AdamW'}

In [ ]:

model.train(
    data=os.path.join('splitted_dataset', 'train_data.yaml'),
    epochs=30,
    imgsz=640,
    lr0=best_params["lr0"],
    lrf=best_params["lrf"],
    momentum=best_params["momentum"],
    weight_decay=best_params["weight_decay"],
    batch=best_params["batch"],
    optimizer = best_params["optimizer"],

    name='fine_tuned_hybrid',
    project='/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_ConvNext/optuna_full',
)

Ultralytics 8.3.239 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=splitted_dataset/train_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=2.8135802639623318e-05, lrf=0.3575501730158067, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.8114575443568205, mosaic=1.0, multi_scale=False, name=fine_tuned_hybrid2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, over

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f1eb5f5d520>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
import torch
torch.save(model.model.state_dict(), "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_ConvNext/full_model.pth")
# torch.save({
#     "model": model.model.state_dict(),
#     "optimizer": model.trainer.optimizer.state_dict(),
#     "epoch": model.trainer.epoch
# }, "/content/drive/MyDrive/Multi Encoder Vehicle Detection/results/Hybrid_MobileSam_Clip_unfreezed/full_model.pth")
